In [ ]:
'''
Deep learning workflow

1. Data
2. Create a model
3. Optimize model parameter (finding best weights)
4. Save the trained model
'''

'\nDeep learning workflow\n\n1. Data\n2. Create a model\n3. Optimize model parameter (finding best weights)\n4. Save the trained model\n'

In [ ]:
import torch #pytorch library that helps in building deep learning algorithms
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

## Download the data

In [ ]:
training_data = datasets.FashionMNIST(root = "data", train = True, download = True, transform = ToTensor())
test_data = datasets.FashionMNIST(root = "data", train = False, download = True, transform = ToTensor())


100%|██████████| 26.4M/26.4M [00:01<00:00, 13.7MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 202kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.79MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 14.8MB/s]


In [ ]:
training_data

Dataset FashionMNIST
    Number of datapoints: 60000
    Root location: data
    Split: Train
    StandardTransform
Transform: ToTensor()

In [ ]:
test_data

Dataset FashionMNIST
    Number of datapoints: 10000
    Root location: data
    Split: Test
    StandardTransform
Transform: ToTensor()

## Batching of data

In [ ]:
batch_size = 90

train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

In [ ]:
for X,y in test_dataloader: # Image -- colour image shape(batch_size, number of channel, length, width)
   print(X.shape)           # Image -- black and white - no of channels will be 1
   print(y.shape)
  #  print(y)
   break

torch.Size([90, 1, 28, 28])
torch.Size([90])


## Create the model

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"using {device} device")

using cuda device


In [ ]:
class NeuralNetwork(nn.Module):

  def __init__(self): #declare the architecture
    super().__init__()

    self.flatten = nn.Flatten()
    self.linear1 = nn.Linear(28*28, 512)
    self.linear2 = nn.Linear(512, 512)
    self.linear3 = nn.Linear(512, 10)
    self.relu = nn.ReLU()


  def forward(self, x):# is always used to passthe inputs to the nn

    x = self.flatten(x)
    x = self.linear1(x)
    x = self.relu(x)
    x = self.linear2(x)
    x = self.relu(x)
    x = self.linear3(x)
    return x

In [ ]:
model = NeuralNetwork()
model = model.to(device)

## Optimization - Gradient descent + backpropagation

In [ ]:
loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.SGD(model.parameters(), lr = 1e-3)

In [ ]:
# Batch of the data/ Pass it to the model/ Compute loss function/ Update the weights

def train(dataloader, model, loss_fn, optimizer):

  model.train()
  for batch, (X,y) in enumerate(dataloader):
    X = X.to(device)
    y = y.to(device)

    #compute predictions
    pred = model(X)

    # compute loss function
    loss = loss_fn(pred, y)


    # Bacpropagation
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()


    if batch % 120 == 0:
      loss = loss.item()
      print(f"loss: {loss: >7f}")

In [ ]:
def test(dataloader, model, loss_fn):

  model.eval()

  num_batches = len(dataloader)

  test_loss, correct = 0, 0

  with torch.no_grad():

    for X, y in dataloader:
      X = X.to(device)
      y = y.to(device)

      #compute predictions
      pred = model(X)

      # compute loss function
      test_loss = loss_fn(pred, y)


      correct += (pred.argmax(1) == y).type(torch.float).sum().item()

  test_loss = test_loss / num_batches
  correct = correct / len(dataloader.dataset)

  print(f"Test Accuracy {100 * correct}, Avg_loss:{test_loss}")



## Training Phase

In [ ]:
epochs = 7

for t in range(7):
  print(f"Epoch {t+1}")
  train(train_dataloader, model, loss_fn, optimizer)
  test(test_dataloader, model, loss_fn)

print("Done")

Epoch 1
loss: 2.308348
loss: 2.289755
loss: 2.272598
loss: 2.262645
loss: 2.245170
loss: 2.215650
Test Accuracy 35.79, Avg_loss:0.02007421851158142
Epoch 2
loss: 2.211693
loss: 2.182276
loss: 2.164463
loss: 2.167883
loss: 2.132830
loss: 2.082844
Test Accuracy 48.730000000000004, Avg_loss:0.019101817160844803
Epoch 3
loss: 2.083679
loss: 2.035817
loss: 2.008322
loss: 2.012314
loss: 1.949179
loss: 1.871937
Test Accuracy 60.81999999999999, Avg_loss:0.017228595912456512
Epoch 4
loss: 1.874418
loss: 1.807832
loss: 1.768272
loss: 1.769579
loss: 1.690579
loss: 1.593939
Test Accuracy 63.73, Avg_loss:0.014546755701303482
Epoch 5
loss: 1.602257
loss: 1.537523
loss: 1.500202
loss: 1.517666
loss: 1.447490
loss: 1.348496
Test Accuracy 63.690000000000005, Avg_loss:0.012147028930485249
Epoch 6
loss: 1.369547
loss: 1.318188
loss: 1.290993
loss: 1.329499
loss: 1.269095
loss: 1.172663
Test Accuracy 64.3, Avg_loss:0.010321464389562607
Epoch 7
loss: 1.202884
loss: 1.161363
loss: 1.144227
loss: 1.201736
lo

In [ ]:
torch.save(model.state_dict(), "/content/weights/iteration1.pth")
print("saved")

RuntimeError: Parent directory /content/weights does not exist.

In [ ]:
model = NeuralNetwork().to(device)
model.load_state_dict(torch.load("/content/weights/iteration1.pth"))

FileNotFoundError: [Errno 2] No such file or directory: '/content/weights/iteration1.pth'